[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S06/S06_03_preprocesado_pipeline.ipynb)

# S06 · 03 — Preprocesado y *pipelines* reproducibles

**Tiempo estimado:** 55 min · **Sprint 6** · M2 (UD 2.5) + M3 (UD 3.1)

**Contexto TurisData.** Ya sabemos limpiar `reservas_sucio.csv`. Ahora hay que **preparar
las columnas para un modelo**: rellenar nulos, poner las variables en la misma escala y
convertir las categorías en números. Si lo hacemos a mano, tarde o temprano cometeremos
un error de fuga. La solución de scikit-learn es el **`Pipeline`**: una receta única,
ajustada solo con los datos de entrenamiento y aplicable a datos nuevos.

**Al terminar sabrás:**
1. Separar entrenamiento y prueba correctamente (estratificando).
2. Imputar, escalar y codificar con `SimpleImputer`, `StandardScaler` y `OneHotEncoder`.
3. Juntarlo todo con `ColumnTransformer` y `Pipeline`.
4. Explicar qué es la **fuga de datos** en el preprocesado y cómo se evita.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.base import clone

SEMILLA = 42
sns.set_theme(style="whitegrid")

## 1. Punto de partida: datos ya limpiados

La función `limpiar_reservas` es la de la práctica anterior (aquí te la damos hecha).
Solo aplica **reglas fijas** (no aprende nada de los datos), así que puede ejecutarse
antes de separar train/test sin riesgo de fuga.

In [ ]:
EQUIV_PAIS = {"es": "ES", "españa": "ES", "uk": "UK", "reino unido": "UK", "de": "DE",
              "nl": "NL", "it": "IT", "fr": "FR", "nordicos": "Nordicos", "otros": "otros"}

def limpiar_reservas(tabla):
    t = tabla.drop_duplicates().reset_index(drop=True).copy()
    t["precio_noche"] = pd.to_numeric(t["precio_noche"].astype(str).str.replace(",", "."), errors="coerce")
    iso = pd.to_datetime(t["fecha_reserva"], format="%Y-%m-%d", errors="coerce")
    esp = pd.to_datetime(t["fecha_reserva"], format="%d/%m/%Y", errors="coerce")
    t["fecha_reserva"] = iso.fillna(esp)
    t["fecha_llegada"] = pd.to_datetime(t["fecha_llegada"], format="%Y-%m-%d")
    t["canal"] = t["canal"].str.strip().str.lower().str.replace(" ", "_")
    t["pais_origen"] = t["pais_origen"].str.strip().str.lower().map(EQUIV_PAIS)
    t["precio_noche"] = t["precio_noche"].where(t["precio_noche"].between(20, 1000))
    t["noches"] = t["noches"].where(t["noches"].between(1, 60))
    t["antelacion_dias"] = (t["fecha_llegada"] - t["fecha_reserva"]).dt.days
    t["mes_llegada"] = t["fecha_llegada"].dt.month
    return t

reservas = limpiar_reservas(pd.read_csv(dato("reservas_sucio.csv")))
print(reservas.shape)
reservas.isna().sum()[lambda s: s > 0]

## 2. Separar entrenamiento y prueba

**Idea.** Guardamos una parte de los datos (p. ej. 20 %) que el modelo **nunca verá** durante
el ajuste: es el examen final. Con `stratify=y` mantenemos la misma proporción de
canceladas en ambos conjuntos. `random_state` fija la semilla para poder repetirlo.

In [ ]:
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "mes_llegada"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X = reservas[columnas_num + columnas_cat]
y = reservas["cancelada"]

**Ejercicio 1.** Separa `X` e `y` en `X_train, X_test, y_train, y_test` con un 20 % para
prueba, `random_state=SEMILLA` y estratificando por `y`.

In [ ]:
# TODO: train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
...
assert len(X_train) == 1200 and len(X_test) == 300, "Con 1500 filas y test_size=0.2 deben ser 1200 / 300"
assert abs(y_train.mean() - y_test.mean()) < 0.01, "Estratifica con stratify=y para conservar la proporción"

## 3. Escalado

**Idea.** `precio_noche` va de 50 a 250 y `ninos` de 0 a 3. Los modelos basados en
distancias o en gradiente (regresión logística, k-NN, SVM, redes) se "fijan" en las
columnas grandes. Escalar pone todas en una escala comparable:
- `StandardScaler`: resta la media y divide por la desviación (media 0, desviación 1).
- `MinMaxScaler`: lleva todo al rango 0–1 (sensible a valores extremos).

**Se ajusta (`fit`) con train y se aplica (`transform`) a train y a test.**

In [ ]:
imputador = SimpleImputer(strategy="median")
escalador = StandardScaler()
num_train = escalador.fit_transform(imputador.fit_transform(X_train[columnas_num]))
num_test = escalador.transform(imputador.transform(X_test[columnas_num]))
print("Media de cada columna en train (≈0):", num_train.mean(axis=0).round(2))
print("Media en test (no tiene por qué ser 0):", num_test.mean(axis=0).round(2))

**Ejercicio 2.** Crea un `MinMaxScaler` llamado `esc_minmax`, ajústalo **solo con
`X_train[["precio_noche"]].dropna()`** y guarda en `precio_test_esc` el precio de test
transformado (rellena antes los nulos de test con la mediana de train: `mediana_precio`).

In [ ]:
# TODO: fit con train; transform con test; la mediana también sale de train
...
assert esc_minmax.data_min_[0] == X_train["precio_noche"].min(), "El escalador debe ajustarse con train, no con test ni con todo"
assert precio_test_esc.shape == (300, 1)
assert mediana_precio == X_train["precio_noche"].median(), "La mediana debe salir de train"

## 4. Codificación de categóricas

**Idea.** Los modelos solo entienden números.
- **One-hot**: una columna 0/1 por cada categoría. Para categorías **sin orden** (canal, país).
  Con `handle_unknown="ignore"` no falla si en test aparece una categoría nueva.
- **Ordinal**: 0, 1, 2… respetando un **orden real** (depósito: sin < parcial < total).

No uses ordinal en categorías sin orden: el modelo creería que "UK" está "entre" "ES" y "DE".

In [ ]:
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehot.fit(X_train[["canal"]])
print(onehot.get_feature_names_out())

**Ejercicio 3.** Codifica `deposito` con un `OrdinalEncoder` que respete el orden
`["sin_deposito", "parcial", "total"]`. Guarda el encoder en `ord_dep` y el resultado
de train en `dep_train_cod`.

In [ ]:
# TODO: OrdinalEncoder(categories=[[...]]) y fit_transform sobre X_train[["deposito"]]
...
assert set(np.unique(dep_train_cod)) == {0.0, 1.0, 2.0}, "Deben salir los valores 0, 1 y 2"
assert ord_dep.transform(pd.DataFrame({"deposito": ["total"]}))[0, 0] == 2.0, "'total' debe valer 2"

## 5. `ColumnTransformer` y `Pipeline`: la receta completa

**Idea.** Cada tipo de columna necesita su propio tratamiento:
- numéricas → imputar mediana → escalar,
- categóricas → imputar la más frecuente → one-hot.

`ColumnTransformer` aplica cada receta a sus columnas y `Pipeline` encadena el preprocesado
con el modelo. Al llamar a `.fit(X_train, y_train)` **todo** se ajusta solo con train.

**Ejercicio 4.** Construye `preprocesado` (un `ColumnTransformer`) con dos ramas:
`"num"` (imputador de mediana + `StandardScaler`) sobre `columnas_num`, y `"cat"`
(imputador `most_frequent` + `OneHotEncoder(handle_unknown="ignore")`) sobre `columnas_cat`.

In [ ]:
# TODO: ColumnTransformer([("num", Pipeline([...]), columnas_num), ("cat", Pipeline([...]), columnas_cat)])
...
matriz = preprocesado.fit_transform(X_train)
assert matriz.shape[0] == 1200, "Debe transformar las 1200 filas de train"
assert not np.isnan(matriz if isinstance(matriz, np.ndarray) else matriz.toarray()).any(), "Tras el preprocesado no deben quedar nulos"
assert matriz.shape[1] > len(columnas_num) + len(columnas_cat), "Las categóricas deben expandirse en columnas one-hot"
print("Columnas tras el preprocesado:", matriz.shape[1])

**Ejercicio 5.** Crea `modelo`, un `Pipeline` con los pasos `("prep", preprocesado)` y
`("clf", LogisticRegression(max_iter=1000, random_state=SEMILLA))`. Ajústalo con train y
guarda en `acc_test` su exactitud (`score`) en test.

In [ ]:
# TODO: Pipeline([("prep", preprocesado), ("clf", LogisticRegression(...))]).fit(X_train, y_train)
...
assert hasattr(modelo, "predict"), "modelo debe ser un Pipeline"
assert acc_test > 0.60, "La exactitud debería superar 0,60; revisa el pipeline"
print(f"Exactitud en test: {acc_test:.3f}  (referencia «siempre la clase mayoritaria»: {max(y_test.mean(), 1 - y_test.mean()):.3f})")

## 6. Fuga de datos en el preprocesado

**Idea.** Hay **fuga** cuando el modelo "ve" información del conjunto de prueba durante el
ajuste. Un caso muy típico: escalar o imputar **todo** el dataset antes de separar.
Los estadísticos (media, mediana) ya contienen información de test. El efecto suele ser
pequeño en un escalado, pero el hábito es peligroso: con otras transformaciones puede
inflar mucho las métricas. Con un `Pipeline` el problema desaparece.

In [ ]:
# MAL: el escalador "ve" también las filas de test
esc_mal = StandardScaler().fit(reservas[["precio_noche"]].fillna(reservas["precio_noche"].median()))
# BIEN: solo train
esc_bien = StandardScaler().fit(X_train[["precio_noche"]].fillna(X_train["precio_noche"].median()))
print("Media aprendida (mal, con todo):", esc_mal.mean_[0].round(3))
print("Media aprendida (bien, solo train):", esc_bien.mean_[0].round(3))

## 7. Un vistazo a la reducción de dimensionalidad (PCA)

Con muchas columnas (tras el one-hot pueden ser decenas), **PCA** las resume en pocas
"componentes" que conservan la mayor parte de la variación. No es obligatorio aquí; sirve
para visualizar y para comprimir. Solo con columnas numéricas y ya escaladas.

In [ ]:
pca = PCA(n_components=5, random_state=SEMILLA).fit(num_train)
plt.figure(figsize=(6, 3.5))
plt.bar(range(1, 6), pca.explained_variance_ratio_, color="#2a6f97")
plt.title("Varianza explicada por componente (PCA)")
plt.xlabel("Componente")
plt.ylabel("Proporción de la varianza")
plt.show()

## 8. Mini-reto: comprueba que tu pipeline es reproducible

Un pipeline reproducible da **las mismas predicciones** si lo vuelves a ajustar con los
mismos datos. Crea `modelo2` (otra instancia del mismo pipeline usando `clone`), ajústalo
y guarda en `son_iguales` un booleano que diga si las predicciones de test coinciden con
las de `modelo`.

In [ ]:
# TODO: usa clone(modelo).fit(...) y compara las predicciones con np.array_equal
...
assert son_iguales, "Las predicciones deberían coincidir: comprueba que fijas la semilla y ajustas con los mismos datos"
assert modelo2 is not modelo, "modelo2 debe ser una instancia nueva (usa clone)"

## 9. Para reflexionar

1. ¿Por qué imputamos con la mediana de **train** y no con la de todo el conjunto?
2. ¿Qué pasa si en test aparece un país que no estaba en train? ¿Qué hace `handle_unknown="ignore"`?
3. ¿Cuándo usarías `MinMaxScaler` en vez de `StandardScaler`? ¿Qué le pasa a MinMax con un precio de 9999 €?
4. ¿Qué ventaja tiene guardar todo el preprocesado dentro del pipeline cuando el cliente quiera usar el modelo en producción?

## 10. Resumen: qué has aprendido
- `train_test_split` con estratificación y semilla.
- Imputar, escalar y codificar siempre **con `fit` solo en train**.
- `ColumnTransformer` + `Pipeline` = una receta única, reproducible y sin fuga.